# Whisper-large-v3 → Yorùbá LoRA fine-tune (v4)

Step-by-step training notebook. Same proven hyperparams as `Whisper.ipynb` (A100 80 GB profile — batch 48, Unsloth grad ckpt, fused AdamW, bf16). What changes vs. v3:

1. **Strict data split**: reserve `N_HOLDOUT = 50` samples the model *never* sees, then 80% train / 20% eval on the rest.
2. **Reproducible splits**: deterministic via `SEED`. Holdout indices + payload saved to Drive, so `Whisper_test.ipynb` can evaluate on the exact same held-out clips later.
3. **Drive-first artifact storage**: config, splits, training loss history, eval metrics, **LoRA adapter, and merged 16-bit checkpoint** all written to `MyDrive/yoruba-pipeline-logs/training/<run-id>/`. No HF push from this notebook — that's a manual decision later, once you've evaluated the run from Drive.
4. **Step-by-step cells**: each cell is one discrete operation with a markdown header. Re-run any step independently.

**Why a real holdout matters**: the previous v3 fine-tune evaluated on FLEURS yo_ng (out-of-distribution from Hidi-agili training data), but never on Hidi-agili clips the model didn't see. The 50-sample holdout gives an in-distribution overfitting check that's independent from the FLEURS benchmark.

## Step 1 — Hugging Face auth + Google Drive mount (handle prompts NOW)\n\n**This is the only cell that prompts you.** Authorize Drive + HF auth in the popup that appears, then hit "Run All" again (or let it continue) and you can walk away — the rest of the notebook runs unattended for ~30–40 min.\n\nDrive prompts via a popup the first time per session; subsequent runs in the same session are silent. HF auth uses the `HF_TOKEN` Colab Secret if set (silent), otherwise it falls back to env var.

In [ ]:
import os
from pathlib import Path

# --- Drive first: triggers the popup; user authorizes, then walks away ---
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/yoruba-pipeline-logs/training")
    print(f"Drive mounted → {DRIVE_ROOT}")
except Exception as e:
    DRIVE_ROOT = Path("./yoruba-pipeline-logs/training")
    print(f"Drive unavailable ({type(e).__name__}); using local fallback → {DRIVE_ROOT}")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# --- HF auth: silent if HF_TOKEN secret is set in Colab ---
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF auth: ok")
else:
    print("HF auth: skipped (no HF_TOKEN secret). Push_to_hub will be disabled regardless.")

print("\nAuth done. The rest of the notebook is hands-off — you can walk away.")

## Step 2 — Install dependencies\n\nHeavy install (~3–4 min). Runs after auth so you can leave the notebook unattended from here on.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install librosa soundfile evaluate jiwer torchcodec "datasets>=3.4.1,<4.0.0"

## Step 3 — Run config (single source of truth)

Everything that affects the run lives here. Anything changed below this cell breaks reproducibility — change here and re-run from the top.

In [ ]:
import datetime as _dt

# --- Identity ---
RUN_ID    = _dt.datetime.utcnow().strftime("%Y-%m-%dT%H-%M-%SZ")
REPO_BASE = "devalade/whisper-large-v3-yoruba-v4"   # HF Hub destination

# --- Data ---
DATASET_REPO  = "Hidi-agili/yoruba_tts_dataset"
DATASET_SPLIT = "train"
N_HOLDOUT     = 50           # samples the model NEVER sees — held out for later eval
EVAL_FRAC     = 0.20         # of the remaining (post-holdout) data
SEED          = 3407

# --- Model ---
BASE_MODEL = "unsloth/whisper-large-v3"
LANGUAGE   = "yoruba"
TASK       = "transcribe"

# --- LoRA (matches the v3 colab recipe that delivered the +38 pp Y-WER-perm win) ---
LORA_R              = 64
LORA_ALPHA          = 64
LORA_TARGETS        = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]
USE_GRAD_CKPT       = "unsloth"

# --- Trainer ---
PER_DEVICE_BATCH    = 48
GRAD_ACCUM          = 1
NUM_EPOCHS          = 8       # bumped from 3 — early stopping below caps wasted compute
EARLY_STOP_PATIENCE = 2       # stop if eval WER doesn't improve for 2 consecutive epochs
SAVE_TOTAL_LIMIT    = 4       # ≥ patience+1 so the best checkpoint can't get evicted
LEARNING_RATE       = 1e-4
WARMUP_RATIO        = 0.05
LR_SCHEDULER        = "cosine"
WEIGHT_DECAY        = 0.001
OPTIM               = "adamw_torch_fused"

RUN_DIR = DRIVE_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_ID  = {RUN_ID}")
print(f"RUN_DIR = {RUN_DIR}")
print(f"DEST    = {REPO_BASE}")
print(f"epochs  = up to {NUM_EPOCHS} (early-stop patience={EARLY_STOP_PATIENCE})")

## Step 4 — Load dataset, reserve holdout, split 80/20

Deterministic order: shuffle the full dataset with `SEED`, take the first `N_HOLDOUT` rows as the **holdout** (the model never sees these), then split the remainder 80/20 into train/eval.

Three resulting datasets:

- `holdout_ds`  — `N_HOLDOUT` clips for offline evaluation later.
- `train_ds`    — 80% of the rest (model trains on these).
- `eval_ds`     — 20% of the rest (per-epoch eval during training).

In [ ]:
from datasets import load_dataset, Audio

full = load_dataset(DATASET_REPO, split=DATASET_SPLIT)
full = full.cast_column("audio", Audio(sampling_rate=16000))
full = full.shuffle(seed=SEED)

N = len(full)
assert N > N_HOLDOUT + 100, f"dataset too small ({N}) for N_HOLDOUT={N_HOLDOUT}"

holdout_idx_in_shuffled = list(range(N_HOLDOUT))
rest_idx                 = list(range(N_HOLDOUT, N))

holdout_ds = full.select(holdout_idx_in_shuffled)
rest       = full.select(rest_idx)

split = rest.train_test_split(test_size=EVAL_FRAC, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

print(f"full     = {N}")
print(f"holdout  = {len(holdout_ds)} (never seen)")
print(f"train    = {len(train_ds)} (80% of rest)")
print(f"eval     = {len(eval_ds)} (20% of rest)")
print(f"covered  = {len(holdout_ds) + len(train_ds) + len(eval_ds)} / {N}")

## Step 5 — Persist holdout + config to Drive

Three artifacts:

- `config.json` — full run config (model, hyperparams, split sizes, seed). Lets you reproduce the exact recipe months later.
- `holdout/` — HF dataset on disk with audio + text. `Whisper_test.ipynb` (or any eval script) can `load_from_disk()` this directly.
- `holdout_meta.jsonl` — text + duration per clip, easy to skim outside Python.

Drive write of 50 audio clips ≈ 5–10 MB. Trivial.

In [ ]:
import json

config_payload = {
    "run_id":          RUN_ID,
    "timestamp_utc":   _dt.datetime.utcnow().isoformat() + "Z",
    "repo_base":       REPO_BASE,
    "dataset":         {
        "repo":  DATASET_REPO,
        "split": DATASET_SPLIT,
        "size":  N,
    },
    "splits": {
        "holdout": len(holdout_ds),
        "train":   len(train_ds),
        "eval":    len(eval_ds),
        "eval_frac": EVAL_FRAC,
        "seed":      SEED,
    },
    "base_model":      BASE_MODEL,
    "lora": {
        "r":              LORA_R,
        "alpha":          LORA_ALPHA,
        "targets":        LORA_TARGETS,
        "grad_ckpt":      USE_GRAD_CKPT,
    },
    "trainer": {
        "per_device_batch":     PER_DEVICE_BATCH,
        "grad_accum":           GRAD_ACCUM,
        "num_epochs":           NUM_EPOCHS,
        "early_stop_patience":  EARLY_STOP_PATIENCE,
        "save_total_limit":     SAVE_TOTAL_LIMIT,
        "learning_rate":        LEARNING_RATE,
        "warmup_ratio":         WARMUP_RATIO,
        "lr_scheduler":         LR_SCHEDULER,
        "weight_decay":         WEIGHT_DECAY,
        "optim":                OPTIM,
    },
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as f:
    json.dump(config_payload, f, ensure_ascii=False, indent=2)
print(f"wrote {RUN_DIR/'config.json'}")

holdout_dir = RUN_DIR / "holdout"
holdout_ds.save_to_disk(str(holdout_dir))
print(f"wrote {holdout_dir} ({len(holdout_ds)} clips)")

with (RUN_DIR / "holdout_meta.jsonl").open("w", encoding="utf-8") as f:
    for i, row in enumerate(holdout_ds):
        f.write(json.dumps({
            "i":          i,
            "text":       row["text"],
            "duration_s": round(len(row["audio"]["array"]) / row["audio"]["sampling_rate"], 3),
        }, ensure_ascii=False) + "\n")
print(f"wrote {RUN_DIR/'holdout_meta.jsonl'}")

## Step 6 — Load base model + apply LoRA

Same recipe as the v3 colab run: rank-64 LoRA over all attention + MLP projections, with Unsloth gradient checkpointing on. Trainable params land around 115 M (≈7% of the base).

In [ ]:
# --- Workaround for transformers 4.56.2 + huggingface_hub mismatch ---
# AutoTokenizer.from_pretrained iterates `list_repo_templates(...)` which queries
# an `additional_chat_templates/` directory most model repos don't have. Newer
# huggingface_hub raises 404; transformers 4.56.2 doesn't catch it; Unsloth
# then bails with a misleading "tokenizer weirdly not loaded" error.
# Patch the lookup to return [] on 404 before loading.
import transformers.utils.hub as _hub_utils
from huggingface_hub.errors import EntryNotFoundError, HfHubHTTPError

if not getattr(_hub_utils, "_unsloth_template_patched", False):
    _orig_list_repo_templates = _hub_utils.list_repo_templates
    def _safe_list_repo_templates(*args, **kwargs):
        try:
            return list(_orig_list_repo_templates(*args, **kwargs))
        except (EntryNotFoundError, HfHubHTTPError):
            return []
    _hub_utils.list_repo_templates = _safe_list_repo_templates
    _hub_utils._unsloth_template_patched = True
    print("patched list_repo_templates to tolerate 404")

# --- Now load the base model + apply LoRA ---
from unsloth import FastModel
from transformers import WhisperForConditionalGeneration
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL,
    dtype=None,
    load_in_4bit=False,
    auto_model=WhisperForConditionalGeneration,
    whisper_language=LANGUAGE.capitalize(),
    whisper_task=TASK,
)

model = FastModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGETS,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=USE_GRAD_CKPT,
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
    task_type=None,   # ** MUST be None for Whisper **
)

model.generation_config.language = "<|yo|>"
model.generation_config.task = TASK
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None
print("base model + LoRA ready")

## Step 7 — Feature extraction (batched, single-process)

`batched=True, num_proc=1` — see `docs/whisper-data-prep-gotchas.md` for why `num_proc>1` deadlocks on HF Audio columns.

In [ ]:
feature_extractor = tokenizer.feature_extractor
text_tokenizer    = tokenizer.tokenizer

def prepare_batch(batch):
    arrays = [a["array"] for a in batch["audio"]]
    sr     = batch["audio"][0]["sampling_rate"]
    feats  = feature_extractor(arrays, sampling_rate=sr)
    labels = text_tokenizer(batch["text"]).input_ids
    return {"input_features": feats.input_features, "labels": labels}

train_prepared = train_ds.map(
    prepare_batch, batched=True, batch_size=32,
    remove_columns=train_ds.column_names, desc="train",
)
eval_prepared = eval_ds.map(
    prepare_batch, batched=True, batch_size=32,
    remove_columns=eval_ds.column_names, desc="eval",
)
print(f"train_prepared = {len(train_prepared)}  eval_prepared = {len(eval_prepared)}")

## Step 8 — Metric (WER) + data collator

In [ ]:
import evaluate
import numpy as np
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

wer_metric = evaluate.load("wer")

def preprocess_logits_for_metrics(logits, labels):
    """Argmax in-place at the end of every eval forward pass so the trainer
    accumulates int token IDs instead of (batch, seq, vocab=51865) fp32
    logits. Cuts eval-memory by ~30,000×. Mandatory for Whisper eval at
    realistic batch sizes — without this, eval OOMs on any GPU."""
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics(pred):
    pred_ids  = pred.predictions   # already argmaxed by preprocess_logits_for_metrics
    label_ids = pred.label_ids
    # Trainer pads BOTH preds and labels with -100 across batches so it can
    # concat variable-length sequences. Replace -100 with pad_token_id before
    # decoding; otherwise the Rust tokenizer hits OverflowError trying to cast
    # negative ints to u32.
    pred_ids[pred_ids == -100]   = tokenizer.pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

## Step 9 — Build the trainer

Same A100 80 GB profile as Whisper.ipynb. Checkpoints land under `outputs/` in `/content`; we'll back the best one up to Drive after training.

In [ ]:
import os as _os
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback
from unsloth import is_bf16_supported

DL_WORKERS = max(2, (_os.cpu_count() or 4) // 2)

# Step-based eval — gives intra-epoch learning signal without spending too
# much of the wall-clock budget inside the eval loop.
EVAL_EVERY_N_STEPS = 200

args = Seq2SeqTrainingArguments(
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    dataloader_num_workers=DL_WORKERS,
    dataloader_pin_memory=True,
    dataloader_persistent_workers=True,

    num_train_epochs=NUM_EPOCHS,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,

    bf16=is_bf16_supported(),
    fp16=not is_bf16_supported(),
    tf32=True,
    optim=OPTIM,

    logging_steps=2,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY_N_STEPS,
    save_strategy="steps",
    save_steps=EVAL_EVERY_N_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    # Belt-and-suspenders alongside preprocess_logits_for_metrics: offload eval
    # accumulator to CPU every N batches. Cheap insurance against fragmentation.
    eval_accumulation_steps=4,

    remove_unused_columns=False,
    label_names=["labels"],
    seed=SEED,
    output_dir="outputs",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    train_dataset=train_prepared,
    eval_dataset=eval_prepared,
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=tokenizer),
    tokenizer=tokenizer.feature_extractor,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    args=args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
)
print(f"trainer ready — up to {NUM_EPOCHS} epochs, eval every {EVAL_EVERY_N_STEPS} steps, "
      f"early-stop patience {EARLY_STOP_PATIENCE}")

## Step 10 — Train\n\nUp to `NUM_EPOCHS` (8 by default), with early stopping after `EARLY_STOP_PATIENCE` (2) flat evals. ~7–10 min per epoch on A100 80 GB for ~7 k train clips at batch 48 — so worst case ~60–80 min, but early stop usually trims that.

In [ ]:
trainer_stats = trainer.train()
print("training done")
print(f"  runtime = {trainer_stats.metrics['train_runtime']:.1f}s")

## Step 11 — Persist training log + final eval metrics to Drive

In [ ]:
import json

with (RUN_DIR / "train_log.jsonl").open("w", encoding="utf-8") as f:
    for entry in trainer.state.log_history:
        f.write(json.dumps(entry) + "\n")
print(f"wrote {RUN_DIR/'train_log.jsonl'} ({len(trainer.state.log_history)} entries)")

final_metrics = trainer.evaluate()
with (RUN_DIR / "eval_metrics.json").open("w", encoding="utf-8") as f:
    json.dump({
        "train_metrics": trainer_stats.metrics,
        "final_eval":    final_metrics,
    }, f, ensure_ascii=False, indent=2, default=str)
print(f"wrote {RUN_DIR/'eval_metrics.json'}")
print(f"final eval WER on internal 20% split: {final_metrics.get('eval_wer', 'n/a')}")

## Step 11.5 — Eval on the 50-clip holdout (model has NEVER seen these)\n\nMirrors the internal eval but on the held-out set. Gives a quick number for "how does the fine-tune do on in-distribution data the model hasn't memorised?" — independent from the FLEURS YASR-Bench number you'll get later.\n\nPrepares features for the holdout (skipped during training prep since the holdout was deliberately excluded), then calls `trainer.evaluate` with a `holdout` prefix so the metrics keys don't collide.

In [ ]:
holdout_prepared = holdout_ds.map(
    prepare_batch,
    batched=True,
    batch_size=32,
    remove_columns=holdout_ds.column_names,
    desc="holdout",
)

holdout_metrics = trainer.evaluate(
    eval_dataset=holdout_prepared,
    metric_key_prefix="holdout",
)

with (RUN_DIR / "holdout_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(holdout_metrics, f, ensure_ascii=False, indent=2, default=str)

print(f"\nHoldout (N={len(holdout_prepared)}, never seen during training):")
print(f"  holdout_wer  = {holdout_metrics.get('holdout_wer', 'n/a')}")
print(f"  holdout_loss = {holdout_metrics.get('holdout_loss', 'n/a')}")
print(f"\nCompare to internal eval (20% split):")
print(f"  eval_wer     = {final_metrics.get('eval_wer', 'n/a')}")
print(f"  eval_loss    = {final_metrics.get('eval_loss', 'n/a')}")
print(f"\nwrote {RUN_DIR/'holdout_metrics.json'}")

## Step 11.7 — Training curves (for the thesis)\n\nExtracts train loss, eval loss, and eval WER from `trainer.state.log_history` and plots them. Saves both:\n\n- `training_curves.png` (raster, embed in slides/markdown)\n- `training_curves.pdf` (vector, embed in LaTeX/thesis)\n- `training_curves_data.json` (raw numbers, regenerate the plot later in any style)\n\nAll three land in `RUN_DIR` next to `train_log.jsonl`, so the full provenance for one run is in one folder.

In [ ]:
import json
import matplotlib.pyplot as plt

# --- Pull the curves out of log_history ---
train_steps, train_loss = [], []
eval_steps,  eval_loss, eval_wer = [], [], []

for entry in trainer.state.log_history:
    step = entry.get("step")
    if step is None:
        continue
    if "loss" in entry and "eval_loss" not in entry:
        train_steps.append(step)
        train_loss.append(entry["loss"])
    if "eval_loss" in entry:
        eval_steps.append(step)
        eval_loss.append(entry["eval_loss"])
        eval_wer.append(entry.get("eval_wer"))

# --- Persist raw numbers (cheap, lets you re-plot in any style later) ---
curves_payload = {
    "run_id":      RUN_ID,
    "train_steps": train_steps,
    "train_loss":  train_loss,
    "eval_steps":  eval_steps,
    "eval_loss":   eval_loss,
    "eval_wer":    eval_wer,
}
with (RUN_DIR / "training_curves_data.json").open("w", encoding="utf-8") as f:
    json.dump(curves_payload, f, indent=2)

# --- Plot ---
fig, (ax_loss, ax_wer) = plt.subplots(1, 2, figsize=(12, 4.5))

ax_loss.plot(train_steps, train_loss, label="train loss", linewidth=1, alpha=0.6)
ax_loss.plot(eval_steps,  eval_loss,  label="eval loss",  marker="o", linewidth=2)
ax_loss.set_xlabel("step")
ax_loss.set_ylabel("loss")
ax_loss.set_title(f"Loss — {RUN_ID}")
ax_loss.grid(True, alpha=0.3)
ax_loss.legend()

ax_wer.plot(eval_steps, eval_wer, marker="o", color="C2", linewidth=2)
ax_wer.set_xlabel("step")
ax_wer.set_ylabel("eval WER (%)")
ax_wer.set_title("Eval WER (lower = better)")
ax_wer.grid(True, alpha=0.3)
# Annotate the best-WER point — that's the checkpoint load_best_model_at_end restored
if eval_wer and any(w is not None for w in eval_wer):
    valid = [(s, w) for s, w in zip(eval_steps, eval_wer) if w is not None]
    best_step, best_wer = min(valid, key=lambda sw: sw[1])
    ax_wer.scatter([best_step], [best_wer], s=120, edgecolor="black",
                   facecolor="none", linewidth=1.5, zorder=5)
    ax_wer.annotate(f"best: {best_wer:.2f}%\nstep {best_step}",
                    xy=(best_step, best_wer),
                    xytext=(10, 10), textcoords="offset points",
                    fontsize=9)

fig.tight_layout()
fig.savefig(RUN_DIR / "training_curves.png", dpi=160, bbox_inches="tight")
fig.savefig(RUN_DIR / "training_curves.pdf",         bbox_inches="tight")
plt.show()

print(f"\nwrote:")
print(f"  {RUN_DIR/'training_curves.png'}")
print(f"  {RUN_DIR/'training_curves.pdf'}")
print(f"  {RUN_DIR/'training_curves_data.json'}")

## Step 12 — Save artifacts (local + Drive). HF push deferred.\n\nGoal: train, evaluate, write everything to Drive. **No HF push from this notebook** — that happens later by hand from the Drive backup when you're ready to ship.\n\nFour flags:\n\n- `SAVE_LORA_DRIVE` — copy LoRA adapter to `RUN_DIR/adapter/` (~250 MB).\n- `SAVE_MERGED_LOCAL` — produce the merged 16-bit checkpoint locally (needed for the Drive copy below).\n- `SAVE_MERGED_DRIVE` — copy the merged 16-bit checkpoint to `RUN_DIR/merged_16bit/` (~3 GB). This is what you load for evaluation later; it's the artifact the Whisper_test notebook will pick up.\n- `PUSH_MERGED_HUB` — **off by default**. Flip to `True` when you're ready to push.

In [ ]:
import shutil

SAVE_LORA_DRIVE   = True
SAVE_MERGED_LOCAL = True
SAVE_MERGED_DRIVE = True
PUSH_MERGED_HUB   = False    # ← deferred. Flip to True later when ready to ship.

assert HF_TOKEN or not PUSH_MERGED_HUB, "PUSH_MERGED_HUB=True but no HF_TOKEN — run Step 2 first."

# --- LoRA adapter ---
model.save_pretrained("whisper_yoruba_lora")
tokenizer.save_pretrained("whisper_yoruba_lora")
print("saved LoRA adapter → ./whisper_yoruba_lora")

if SAVE_LORA_DRIVE:
    adapter_dst = RUN_DIR / "adapter"
    if adapter_dst.exists():
        shutil.rmtree(adapter_dst)
    shutil.copytree("whisper_yoruba_lora", str(adapter_dst))
    print(f"backed up adapter → {adapter_dst}")

# --- Merged 16-bit ---
if SAVE_MERGED_LOCAL:
    model.save_pretrained_merged(
        "whisper_yoruba_16bit", tokenizer, save_method="merged_16bit",
    )
    print("saved merged 16-bit → ./whisper_yoruba_16bit")

if SAVE_MERGED_DRIVE:
    assert SAVE_MERGED_LOCAL, "SAVE_MERGED_DRIVE needs SAVE_MERGED_LOCAL=True"
    merged_dst = RUN_DIR / "merged_16bit"
    if merged_dst.exists():
        shutil.rmtree(merged_dst)
    shutil.copytree("whisper_yoruba_16bit", str(merged_dst))
    print(f"backed up merged 16-bit → {merged_dst}")

# --- HF push (off by default) ---
if PUSH_MERGED_HUB:
    model.push_to_hub_merged(
        REPO_BASE, tokenizer, save_method="merged_16bit", token=HF_TOKEN,
    )
    print(f"pushed merged 16-bit → {REPO_BASE}")
else:
    print("HF push skipped (PUSH_MERGED_HUB=False). To ship later:")
    print(f"  model.push_to_hub_merged({REPO_BASE!r}, tokenizer, save_method='merged_16bit', token=HF_TOKEN)")

print(f"\nRun complete. Everything for {RUN_ID} lives at:")
print(f"  {RUN_DIR}")
print("  ├─ config.json")
print("  ├─ holdout/            ← 50 clips the model never saw")
print("  ├─ holdout_meta.jsonl")
print("  ├─ train_log.jsonl")
print("  ├─ eval_metrics.json")
print("  ├─ adapter/            ← LoRA only")
print("  └─ merged_16bit/       ← drop-in for M1_HF_MODEL when you're ready to push")